# nisar-hazard-engine on Colab

Runs the engine's own scan path (tile lookup, NISAR retrieval by HDF5 range reads, P01-P13 preprocessing, every routed pipeline, verdict calibration) on a Colab runtime, with live progress: current step, granule N/M, MB on the wire, MB/s, seconds since the last byte, ETA.

**Before running**

1. Runtime > Change runtime type > CPU (High-RAM if offered). No engine code uses a GPU.
2. Upload `nisar-hazard-engine-colab.zip` to `/content` (left file panel). It already contains your `.env`. No `.env`/token in it? Add `EARTHDATA_TOKEN` under Secrets (key icon) and switch on notebook access, or paste it at the prompt.
3. The Earthdata profile must have "Alaska Satellite Facility Data Access" authorized, otherwise downloads answer 401/403.

`/content` is wiped when the runtime recycles: take results with cell 4 or Drive.

In [ ]:
# 1. Unzip + .env + dependencies + credentials + machine tuning, in one step.
#    If it says "Restart session": Runtime > Restart session, then run this cell again (it skips the install).
import glob, zipfile
from pathlib import Path

DEST = Path("/content/nisar-hazard-engine")
if not any(DEST.rglob("start.py")):
    zips = sorted(glob.glob("/content/*nisar*.zip"))
    if not zips:
        from google.colab import files
        zips = ["/content/" + next(iter(files.upload()))]
    DEST.mkdir(parents=True, exist_ok=True)
    zipfile.ZipFile(zips[0]).extractall(DEST)
START = next(DEST.rglob("colab/start.py"))
%run $START

In [ ]:
# 2. Scan one location: the 10 km MGRS tile containing (LON, LAT), every pipeline routed to it.
#    LOOKBACK_DAYS: days of history. 60 finishes sooner than the engine default (None = 120).
#    NISAR calibrated L-band coverage starts 2026-06-17, so deep time series do not exist yet.
LON, LAT = 90.41849, 23.748
AS_OF, LOOKBACK_DAYS = None, 60

state = nc.scan(LON, LAT, as_of=AS_OF, lookback_days=LOOKBACK_DAYS)

In [ ]:
# 3. Results: table (findings first), then each pipeline's evidence layers
nc.show(state)

In [ ]:
# 4. Take the results with you: run.json + every layer as .npy + PNGs, zipped
zip_path = nc.export(state)
nc.download(zip_path)
# nc.to_drive(zip_path)     # or copy to Google Drive instead

## Same thing as a script (output streams live)

```
!python -u /content/nisar-hazard-engine/colab/scan.py --lon 90.41849 --lat 23.748 --lookback 60
```

Writes `/content/nisar_out.zip`. Run it from a fresh runtime or after a restart; it does its own setup.

## Iterating without re-downloading

A scan stores its SceneBundle under `NISAR_DATA_DIR/bundles`. `rerun` executes pipelines on that bundle with no retrieval:

In [ ]:
# allowed params: coherence_threshold, significance_k, detection_floor_m, far_field_margin
state2 = nc.rerun(state["bundle_id"], ["earthquake_coseismic"], {"significance_k": 2.5})
nc.show(state2)

## Several locations

Scans run one at a time (the runner is single-flight by design: each run holds several hundred MB of arrays). The chunk cache on local disk is shared, so overlapping granules are not downloaded twice.

In [ ]:
points = [(90.41849, 23.748), (-118.5, 34.0)]       # (lon, lat) or (lon, lat, as_of)
states = nc.scan_many(points, lookback_days=60)
for s in states:
    print(s.get("tile"), s["status"])
    display(nc.summary(s).head(5))

## Use your LOCAL web UI with Colab's network

Colab's datacenter link to NASA is far faster than a home connection. Start the engine API here behind a Cloudflare tunnel, then point the local UI at it. The UI shows the same live progress panel.

1. Run the cell below; copy the printed `https://...trycloudflare.com` URL.
2. On your machine, open `http://localhost:3000`, click the engine pill in the nav bar (shows `local` by default), paste the URL and press **Link**. No restart needed. (Alternative: `NEXT_PUBLIC_ENGINE_API=<that url>` in `app-ui/.env.local`, then restart `npm run dev`.)

**The URL has no authentication and scans are enabled: anyone who has it can spend your Earthdata token.** Keep it private and run `nc.stop_serve()` when done.

In [ ]:
url = nc.serve(8900, tunnel=True)       # tunnel=False keeps it inside the runtime (127.0.0.1)
print("NEXT_PUBLIC_ENGINE_API=" + url)
# nc.stop_serve()

## Optional: offline self-test (~3 min, needs no credentials)

In [ ]:
nc.selftest()